In [1]:
import pandas as pd
import os

In [3]:
df = pd.read_csv("data/raw/CIC-ToN-IoT.csv")

In [4]:
print("Dataset shape:", df.shape)
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])
print("Label column: Attack")
print("Number of classes:", df["Attack"].nunique())

Dataset shape: (5351760, 85)
Total records: 5351760
Total columns: 85
Label column: Attack
Number of classes: 10


In [5]:
preview_df = pd.concat(
    [df.iloc[:, :5], df[["Attack"]]],
    axis=1
)

print("Dataset shape:", df.shape)
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])
print("Label column: Attack")
print("Number of classes:", df["Attack"].nunique())

preview_df.head(5)

Dataset shape: (5351760, 85)
Total records: 5351760
Total columns: 85
Label column: Attack
Number of classes: 10


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Attack
0,177.30.87.144-192.168.1.1-0-0-0,177.30.87.144,0,192.168.1.1,0,Benign
1,167.49.176.28-50.165.192.168-0-0-0,167.49.176.28,0,50.165.192.168,0,Benign
2,230.158.52.59-177.21.192.168-0-0-0,230.158.52.59,0,177.21.192.168,0,Benign
3,183.68.192.168-1.1.192.168-0-0-0,183.68.192.168,0,1.1.192.168,0,Benign
4,183.41.192.168-1.1.192.168-0-0-0,183.41.192.168,0,1.1.192.168,0,Benign


In [7]:
import pandas as pd
import numpy as np

# Clean column names
df.columns = df.columns.str.strip()

LABEL_COL = "Attack"

if LABEL_COL not in df.columns:
    raise ValueError(f"Column '{LABEL_COL}' not found. Available columns: {df.columns.tolist()}")

# Columns that should not be used as model features
non_feature_cols = [
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP",
    "Dst Port",
    "Timestamp",
    "Label"
]

# Keep only columns that exist in the dataset
non_feature_cols = [col for col in non_feature_cols if col in df.columns]

print("Removed non-feature columns:")
print(non_feature_cols)

# Separate label
y = df[LABEL_COL]

# Drop label and non-feature columns
X_raw = df.drop(columns=[LABEL_COL] + non_feature_cols)

# Keep numerical features only
X = X_raw.select_dtypes(include=["int64", "int32", "float64", "float32"]).copy()

# Replace infinite values with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Remove rows with invalid values
valid_index = X.dropna().index
X = X.loc[valid_index]
y = y.loc[valid_index]

# Remove constant features
constant_cols = X.columns[X.nunique() <= 1].tolist()
X = X.drop(columns=constant_cols)

print("\nRemoved constant columns:")
print(constant_cols)

print("\nFinal feature shape:", X.shape)
print("Number of selected features:", X.shape[1])

print("\nSelected features:")
for i, col in enumerate(X.columns, start=1):
    print(f"{i}. {col}")

Removed non-feature columns:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Timestamp', 'Label']

Removed constant columns:
['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'URG Flag Cnt', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Subflow Bwd Pkts']

Final feature shape: (5350583, 69)
Number of selected features: 69

Selected features:
1. Protocol
2. Flow Duration
3. Tot Fwd Pkts
4. Tot Bwd Pkts
5. TotLen Fwd Pkts
6. TotLen Bwd Pkts
7. Fwd Pkt Len Max
8. Fwd Pkt Len Min
9. Fwd Pkt Len Mean
10. Fwd Pkt Len Std
11. Bwd Pkt Len Max
12. Bwd Pkt Len Min
13. Bwd Pkt Len Mean
14. Bwd Pkt Len Std
15. Flow Byts/s
16. Flow Pkts/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Tot
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Tot
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd PSH Flags
32. Fwd Header Len
33. Bwd Header Len
34. Fwd Pkts/s
35. Bwd Pkts/s
36. Pkt Len Min
37. Pkt Len

In [8]:
variation_check = pd.DataFrame({
    "feature": X.columns,
    "nunique": X.nunique().values,
    "zero_ratio": (X == 0).mean().values,
    "std": X.std(numeric_only=True).values
})

variation_check = variation_check.sort_values(
    by=["nunique", "zero_ratio", "std"],
    ascending=[True, False, True]
)

variation_check.head(15)

,feature,nunique,zero_ratio,std
30,Fwd PSH Flags,2,0.994747,0.072287
54,Subflow Fwd Pkts,2,0.771705,0.419734
45,CWE Flag Count,3,0.999998,0.001729
46,ECE Flag Cnt,3,0.999998,0.001729
40,FIN Flag Cnt,3,0.245634,0.464593
0,Protocol,3,0.099782,4.305362
60,Fwd Seg Size Min,8,0.099782,11.225973
41,SYN Flag Cnt,82,0.592438,31.719263
42,RST Flag Cnt,142,0.990028,50.792929
47,Down/Up Ratio,142,0.508503,299.793844


In [10]:
candidate_cols = [
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg",
    "Fwd PSH Flags",
    "CWE Flag Count",
    "ECE Flag Cnt",
    "Down/Up Ratio"
]

candidate_cols = [col for col in candidate_cols if col in X.columns]

for col in candidate_cols:
    print("\nFeature:", col)
    print("Unique values:", X[col].nunique())
    print("Zero ratio:", (X[col] == 0).mean())
    print("Top values:")
    print(X[col].value_counts().head(5))


Feature: Bwd Byts/b Avg
Unique values: 6618
Zero ratio: 0.754442459821668
Top values:
Bwd Byts/b Avg
0       4036707
164      308441
192      301985
4205      16672
178       15232
Name: count, dtype: int64

Feature: Bwd Pkts/b Avg
Unique values: 608
Zero ratio: 0.754442459821668
Top values:
Bwd Pkts/b Avg
0    4036707
4     755824
5     343931
6     121452
7      30739
Name: count, dtype: int64

Feature: Bwd Blk Rate Avg
Unique values: 313205
Zero ratio: 0.754455729403693
Top values:
Bwd Blk Rate Avg
0           4036778
20500000        684
24000000        660
23428571        617
27428571        614
Name: count, dtype: int64

Feature: Fwd PSH Flags
Unique values: 2
Zero ratio: 0.9947469275777985
Top values:
Fwd PSH Flags
0    5322476
1      28107
Name: count, dtype: int64

Feature: CWE Flag Count
Unique values: 3
Zero ratio: 0.9999975703582208
Top values:
CWE Flag Count
0    5350570
1         12
2          1
Name: count, dtype: int64

Feature: ECE Flag Cnt
Unique values: 3
Zero ratio:

In [11]:
variation_check = pd.DataFrame({
    "feature": X.columns,
    "nunique": X.nunique().values,
    "zero_ratio": (X == 0).mean().values,
    "std": X.std(numeric_only=True).values
})

variation_check = variation_check.sort_values(
    by=["nunique", "zero_ratio", "std"],
    ascending=[True, False, True]
)

features_to_remove = variation_check.head(2)["feature"].tolist()

print("Features removed to obtain 67 features:")
print(features_to_remove)

X_67 = X.drop(columns=features_to_remove)

print("Final number of features:", X_67.shape[1])
print("Final features:")
for i, col in enumerate(X_67.columns, start=1):
    print(f"{i}. {col}")

Features removed to obtain 67 features:
['Fwd PSH Flags', 'Subflow Fwd Pkts']
Final number of features: 67
Final features:
1. Protocol
2. Flow Duration
3. Tot Fwd Pkts
4. Tot Bwd Pkts
5. TotLen Fwd Pkts
6. TotLen Bwd Pkts
7. Fwd Pkt Len Max
8. Fwd Pkt Len Min
9. Fwd Pkt Len Mean
10. Fwd Pkt Len Std
11. Bwd Pkt Len Max
12. Bwd Pkt Len Min
13. Bwd Pkt Len Mean
14. Bwd Pkt Len Std
15. Flow Byts/s
16. Flow Pkts/s
17. Flow IAT Mean
18. Flow IAT Std
19. Flow IAT Max
20. Flow IAT Min
21. Fwd IAT Tot
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Fwd IAT Min
26. Bwd IAT Tot
27. Bwd IAT Mean
28. Bwd IAT Std
29. Bwd IAT Max
30. Bwd IAT Min
31. Fwd Header Len
32. Bwd Header Len
33. Fwd Pkts/s
34. Bwd Pkts/s
35. Pkt Len Min
36. Pkt Len Max
37. Pkt Len Mean
38. Pkt Len Std
39. Pkt Len Var
40. FIN Flag Cnt
41. SYN Flag Cnt
42. RST Flag Cnt
43. PSH Flag Cnt
44. ACK Flag Cnt
45. CWE Flag Count
46. ECE Flag Cnt
47. Down/Up Ratio
48. Pkt Size Avg
49. Fwd Seg Size Avg
50. Bwd Seg Size Avg
51. Bwd B